# EDA: рекомендации банковских продуктов

Ноутбук использует артефакты из `data/eda_artifacts/`, которые строятся кодом из `src/data/build_dataset.py`.

Основная цель исследования: понять, как формулировать задачу именно как **рекомендацию новых продуктов следующего месяца**, а не как предсказание текущего продуктового портфеля.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.data.build_dataset import build_eda_artifacts
from src.utils.config import ProjectConfig

sns.set_theme(style='whitegrid')
config = ProjectConfig()
paths = build_eda_artifacts(config)
paths

In [ ]:
monthly = pd.read_csv(paths['monthly_overview'])
missing = pd.read_csv(paths['missing_profile'])
segments = pd.read_csv(paths['segment_profile'])
penetration = pd.read_csv(paths['product_penetration'])
new_products = pd.read_csv(paths['new_product_dynamics'])

monthly.head()

## 1. Обзор данных и временная структура

Сначала смотрим на покрытие по месяцам, объём клиентских снапшотов и среднее число продуктов на клиента.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
sns.lineplot(data=monthly, x='snapshot_month', y='rows', marker='o', ax=axes[0])
axes[0].set_title('Количество строк по месяцам')
axes[0].tick_params(axis='x', rotation=45)

sns.lineplot(data=monthly, x='snapshot_month', y='unique_clients', marker='o', ax=axes[1])
axes[1].set_title('Уникальные клиенты по месяцам')
axes[1].tick_params(axis='x', rotation=45)

sns.lineplot(data=monthly, x='snapshot_month', y='avg_products_total', marker='o', ax=axes[2])
axes[2].set_title('Среднее число продуктов на клиента')
axes[2].tick_params(axis='x', rotation=45)
plt.tight_layout()

**Вывод.** Данные имеют ярко выраженную временную структуру, поэтому случайный `train_test_split` по строкам здесь недопустим: он приведёт к утечке будущего и завышенной оценке качества.

## 2. Пропуски и грязные значения

Для проекта важны не только пропуски, но и то, что в исходном CSV встречаются пробелы, шум в типах и смешение чисел со строками.

In [ ]:
pivot_missing = missing.pivot(index='feature_name', columns='snapshot_month', values='missing_share')
plt.figure(figsize=(12, 4))
sns.heatmap(pivot_missing, cmap='Reds', annot=False)
plt.title('Доля пропусков по ключевым признакам')
plt.tight_layout()

**Вывод.** До моделирования требуется обязательная очистка строковых значений, приведение типов и устойчивый пайплайн обработки пропусков. Иначе сервис в проде будет нестабилен даже на валидных банковских профилях.

## 3. Динамика продуктового портфеля

Смотрим, какие продукты чаще всего уже есть у клиентов и как меняется проникновение по месяцам.

In [ ]:
top_products = (
    penetration.groupby('product', as_index=False)['penetration_share']
    .mean()
    .sort_values('penetration_share', ascending=False)
    .head(10)
)

subset = penetration[penetration['product'].isin(top_products['product'])]
plt.figure(figsize=(14, 6))
sns.lineplot(data=subset, x='snapshot_month', y='penetration_share', hue='product', marker='o')
plt.title('Динамика проникновения продуктов')
plt.xticks(rotation=45)
plt.tight_layout()

**Вывод.** У продуктов разная базовая частота. Это сразу подсказывает необходимость ranking-постановки: банку важно не просто предсказать все продукты, а отсортировать кандидатов по релевантности для конкретного клиента.

## 4. Сколько новых продуктов появляется по месяцам

Ключевая цель проекта — предсказывать именно **новые** подключения в следующем месяце.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.barplot(data=new_products, x='snapshot_month', y='new_products_total', color='#2a9d8f', ax=axes[0])
axes[0].set_title('Общее число новых продуктов к следующему месяцу')
axes[0].tick_params(axis='x', rotation=45)

sns.lineplot(data=new_products, x='snapshot_month', y='share_with_new_product', marker='o', ax=axes[1])
axes[1].set_title('Доля клиентов с хотя бы одним новым продуктом')
axes[1].tick_params(axis='x', rotation=45)
plt.tight_layout()

**Вывод.** Событие приобретения нового продукта редкое, поэтому классы несбалансированы. Это объясняет, почему обычная accuracy здесь бесполезна, а ranking-метрики top-k подходят лучше.

## 5. Клиентские сегменты

Проверяем, как сегменты различаются по доходу, возрасту и насыщенности продуктами.

In [ ]:
latest_month = segments['snapshot_month'].max()
latest_segments = segments[segments['snapshot_month'] == latest_month].sort_values('clients', ascending=False)
display(latest_segments)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.barplot(data=latest_segments, x='segmento', y='avg_renta', ax=axes[0])
axes[0].set_title('Средний доход по сегментам')
axes[0].tick_params(axis='x', rotation=30)

sns.barplot(data=latest_segments, x='segmento', y='avg_products_total', ax=axes[1])
axes[1].set_title('Среднее число продуктов по сегментам')
axes[1].tick_params(axis='x', rotation=30)
plt.tight_layout()

**Вывод.** Сегменты действительно отличаются по профилю клиентов, поэтому сегмент как минимум полезен в baseline-модели и точно нужен в supervised-модели.

## 6. Итоговые выводы EDA

1. Формулируем задачу как multilabel recommendation/ranking: по снапшоту месяца `t` предсказать продукты, которые клиент добавит в `t+1`.
2. Во время ранжирования нужно обязательно исключать продукты, которые уже есть у клиента в `t`.
3. Валидация должна быть только time-based.
4. Из-за редкости событий основными метриками становятся `MAP@K`, `Precision@K`, `Recall@K`, `NDCG@K`.
5. Практичная модель должна уметь работать с грязными признаками, пропусками и несбалансированностью.